# Can a pulse reverse dephasing?
Compare indistinguishable static frequency offsets and a fluctuating environment. The detuning is an angular frequency. A perfect midpoint π pulse reverses its sign in the accumulated phase. Predict which noise timescales the pulse will reject before drawing the curves.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

sigma_per_ms = 1.2     # rms angular detuning in rad/ms
tau_ms = 1.5           # correlation time in ms
evolution_ms = 2.0
assert sigma_per_ms > 0 and tau_ms>0 and evolution_ms>0

## Integrate the noise covariance
Use a Gaussian stationary process with covariance σ² exp(−|t−s|/τ). The ensemble coherence is exp(−variance of accumulated phase/2). A series near zero avoids subtracting almost equal floating-point numbers.

In [ ]:
def exponents(t,tau,sigma):
    x=np.asarray(t,dtype=float)/tau
    small=x<1e-3
    free=np.where(small,x**2/2-x**3/6+x**4/24,x+np.expm1(-x))
    echo=np.where(small,x**3/12-x**4/32+7*x**5/960,x+4*np.expm1(-x/2)-np.expm1(-x))
    return sigma**2*tau**2*free,sigma**2*tau**2*echo

free,echo=exponents(evolution_ms,tau_ms,sigma_per_ms)
print(f"Contrast at T={evolution_ms:g} ms: free {np.exp(-free):.6f}; echo {np.exp(-echo):.6f}")

## Locate the frequencies that the echo transmits
For harmonic noise the squared Fourier integral of the switching function measures sensitivity. At zero frequency these two filters make opposite predictions. The vertical scale of the right panel is normalised by the square of the observation time.

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(9,3.9),layout="constrained")
ts=np.linspace(0,3*evolution_ms,240)
cf,ce=exponents(ts,tau_ms,sigma_per_ms)
ax[0].plot(ts,np.exp(-cf),label="Free evolution",color="#a23d21")
ax[0].plot(ts,np.exp(-ce),label="Midpoint echo",color="#175868")
ax[0].plot(ts,np.exp(-.5*(sigma_per_ms*ts)**2),"--",label="Static offsets, free",color="#777777")
ax[0].set(xlabel="Total evolution time (ms)",ylabel="Normalised coherence",title="Same noise, two pulse sequences",ylim=(-.03,1.05)); ax[0].legend(fontsize=8)
cycles=np.linspace(0,3,401); wT=2*np.pi*cycles
ff=np.sinc(cycles)**2
fe=np.zeros_like(wT); fe[1:]=16*np.sin(wT[1:]/4)**4/wT[1:]**2
ax[1].plot(cycles,ff,label="Free evolution",color="#a23d21")
ax[1].plot(cycles,fe,label="Midpoint echo",color="#175868")
ax[1].set(xlabel="Noise cycles during T",ylabel="Squared filter / T²",title="Static and time-dependent fields");ax[1].legend(fontsize=8)
plt.show()

## Test the analytic result without using its closed form
Represent the sign function at midpoints of an even time grid. Use the matrix of pair covariances to evaluate the phase variance. Halve the time step and check the convergence. The pulse sign convention may be reversed globally without changing the contrast.

In [ ]:
def numerical_chi(n,flip):
    dt=evolution_ms/n
    t=(np.arange(n)+.5)*dt
    sign=np.where((t>=evolution_ms/2)&flip,-1.,1.)
    cov=sigma_per_ms**2*np.exp(-abs(t[:,None]-t[None,:])/tau_ms)
    return .5*dt**2*(sign@cov@sign)
for n in (80,160,320):
    print(n,numerical_chi(n,False),numerical_chi(n,True))
assert abs(numerical_chi(320,True)-echo) < 2e-4*(1+sigma_per_ms**2*evolution_ms**2)*(1+evolution_ms/tau_ms)
assert 0<=echo<=free+1e-12

## An optional single-spin check with OneSimulation
Use exact matrix exponentiation through a Hermitian eigensystem. This checks the reversal for a constant detuning on one spin; the stochastic variance above describes an ensemble of many spins.

In [ ]:
try:
    from onesim.numerics.linear.matrices.dense_linear_algebra import symmetric_eigensystem
except ImportError:
    print("The independent spin propagator is available in a local OneSimulation environment.")
else:
    E,U=symmetric_eigensystem(np.diag([sigma_per_ms/2,-sigma_per_ms/2]))
    half=(U*np.exp(-1j*E*evolution_ms/2))@U.T
    pulse=np.array([[0,-1j],[-1j,0]])
    initial=np.array([1,1])/np.sqrt(2)
    final=half@pulse@half@initial
    print("Echo return probability for a static detuning:",abs(np.vdot(initial,final))**2)
    assert np.isclose(abs(np.vdot(initial,final))**2,1.)